# AnnNet showcase

A guided tour of the public API. Ten sections, each self-contained, taking you from
the simplest binary graph to the multilayer math API.

This notebook is shipped with empty outputs — run all cells to populate.

**Sections**

1. Hello AnnNet
2. Attributes everywhere
3. Slices for conditions
4. Hyperedges for reactions
5. Multilayer for time / condition
6. Multilayer math
7. Views & subgraphs
8. IO buffet (round-trip via several formats)
9. Backend interop
10. History & replay


In [ ]:
import annnet
print(annnet.__version__ if hasattr(annnet, '__version__') else 'dev')


## 1. Hello AnnNet

The simplest entry point. Construct a directed graph, add a few vertices and edges, look at the shape.


In [ ]:
from annnet import AnnNet

G = AnnNet(directed=True)
G.add_vertices(['A', 'B', 'C', 'D'])
G.add_edges('A', 'B', edge_id='e1', weight=1.0)
G.add_edges('B', 'C', edge_id='e2', weight=2.0)
G.add_edges('C', 'D', edge_id='e3', weight=1.5)
G


In [ ]:
# Shape
print('nv, ne =', G.nv, G.ne)
print('shape =', G.shape)
print('directed =', G.directed)


In [ ]:
# The variadic add_vertices forms
G2 = AnnNet(directed=False)
G2.add_vertices('single')                          # singleton string
G2.add_vertices(['a', 'b', 'c'])                   # list of ids
G2.add_vertices([{'vertex_id': 'd', 'color': 'red'}])  # dict with attrs
G2.add_vertices([('e', {'color': 'blue'})])        # (id, attrs) tuple
print(sorted(G2.vertices()))


In [ ]:
# Iterate edges
for eid in G._col_to_edge.values():
    rec = G._edges[eid]
    print(f'  {eid}: {rec.src} -> {rec.tgt}  weight={rec.weight}')


## 2. Attributes everywhere

Vertex, edge, slice, and graph-level attributes live in narwhals-backed tables.
Bulk variants exist for every setter.


In [ ]:
# Vertex attributes
G.attrs.set_vertex_attrs('A', label='alpha', tier=1)
G.attrs.set_vertex_attrs_bulk({
    'B': {'label': 'beta', 'tier': 1},
    'C': {'label': 'gamma', 'tier': 2},
    'D': {'label': 'delta', 'tier': 2},
})

# Single-attr read
print('A.label =', G.attrs.get_attr_vertex('A', 'label'))
print('B full =', G.attrs.get_vertex_attrs('B'))


In [ ]:
# Edge attributes (bulk). 'kind' / 'weight' / 'directed' are reserved
# (structural columns), so use a different key.
G.attrs.set_edge_attrs_bulk({
    'e1': {'confidence': 0.9, 'interaction': 'activation'},
    'e2': {'confidence': 0.7, 'interaction': 'inhibition'},
    'e3': {'confidence': 0.95, 'interaction': 'activation'},
})

# Filter by attribute
activations = G.attrs.get_edges_by_attr('interaction', 'activation')
print('activation edges:', activations)


In [ ]:
# Graph-level metadata (uns-style)
G.uns['study'] = 'showcase'
G.uns['author'] = 'demo'
print(G.attrs.get_graph_attributes())


In [ ]:
# Reserved keys are rejected — they're part of the structural contract
try:
    G.attrs.set_edge_attrs('e1', source='X')
except ValueError as e:
    print('caught:', e)


## 3. Slices for conditions

Slices are condition / cohort / sample subsets — like obs categories in AnnData, but for vertices and edges together.


In [ ]:
G.slices.add('control', cohort='wt')
G.slices.add('treated', cohort='ko')
print(G.slices.list())


In [ ]:
# Add edges directly into a slice
G.add_edges('A', 'D', edge_id='e_extra', slice='treated', weight=3.0)
print('treated edges:', G.slices.edges('treated'))
print('treated vertices:', G.slices.vertices('treated'))


In [ ]:
# Slice algebra
G.slices.add_edges('control', ['e1', 'e2'])
G.slices.add_edges('treated', ['e2', 'e3', 'e_extra'])

both = G.slices.intersect(['control', 'treated'])
print('intersection (edges):', both['edges'])
union = G.slices.union(['control', 'treated'])
print('union (edges):', sorted(union['edges']))


In [ ]:
# Per-slice weight overrides — useful for condition-specific weights
G.attrs.set_edge_slice_attrs('treated', 'e2', weight=99.0)
print('global e2 weight:', G.attrs.get_effective_edge_weight('e2'))
print('treated e2 weight:', G.attrs.get_effective_edge_weight('e2', slice='treated'))


## 4. Hyperedges for reactions

A hyperedge connects an arbitrary set of vertices. Two flavours:

- **Undirected** — pass a list of members.
- **Directed** — pass `src=[heads]`, `tgt=[tails]`.


In [ ]:
H = AnnNet(directed=False)
H.add_vertices(['Glc', 'ATP', 'G6P', 'ADP'])

# Undirected: a complex of three species
H.add_edges(['Glc', 'ATP', 'G6P'], edge_id='complex_A')

# Directed reaction: hexokinase  Glc + ATP -> G6P + ADP
H.add_edges(src=['Glc', 'ATP'], tgt=['G6P', 'ADP'], edge_id='hexokinase')
print('ne =', H.ne)
for eid in H._edges:
    rec = H._edges[eid]
    print(f'  {eid}: src={set(rec.src) if rec.src else None}, tgt={set(rec.tgt) if rec.tgt else None}')


In [ ]:
# Pretty edge view shows hyperedges with members / head / tail columns
H.views.edges().head(5)


In [ ]:
# Stoichiometry: directly read/write the incidence matrix column
import numpy as np
M = H._matrix
col = H._edges['hexokinase'].col_idx
print('hexokinase column:')
for vid in H.vertices():
    rec_row = H._entities[H._resolve_entity_key(vid)].row_idx
    val = M[rec_row, col]
    print(f'  {vid:>6}: {val:+.2f}')


## 5. Multilayer for time / condition

Aspects partition vertices into layers. A vertex like `A` can live in multiple layers; the
*supra-node* `(A, ('t1',))` is the actual indexed entity. Intra / inter / coupling edges
have explicit roles.


In [ ]:
import warnings
M = AnnNet(directed=False)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    # One aspect 't' with two elementary layers.
    M.layers.set_aspects(['t'], {'t': ['t1', 't2']})

    # Same vertices A and B in both layers.
    M.add_vertices(['A', 'B'], layer={'t': 't1'})
    M.add_vertices(['A', 'B'], layer={'t': 't2'})

    # Intra-layer edges.
    M.add_edges(('A', ('t1',)), ('B', ('t1',)), edge_id='e_t1', weight=1.0)
    M.add_edges(('A', ('t2',)), ('B', ('t2',)), edge_id='e_t2', weight=1.0)

print('aspects:', M.layers.list_aspects())
print('layers:', M.layers.list_layers())


In [ ]:
# Add diagonal couplings (A in t1 <-> A in t2, B in t1 <-> B in t2)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    n_added = M.layers.add_layer_coupling_pairs([(('t1',), ('t2',))])
print('coupling edges added:', n_added)


In [ ]:
# Subgraph for a specific layer
sub_t1 = M.layers.subgraph_from_layer_tuple(('t1',))
print('t1 subgraph: nv=', sub_t1.nv, 'ne=', sub_t1.ne)


## 6. Multilayer math

Supra-adjacency, supra-Laplacian, transition matrix, layer descriptors.
Same `M` graph from section 5.


In [ ]:
import numpy as np

A = M.layers.supra_adjacency()
print('supra-adjacency (', A.shape, '):')
print(A.toarray())


In [ ]:
# Decomposition: A = A_intra + A_inter + A_coup
A_intra = M.layers.build_intra_block().toarray()
A_inter = M.layers.build_inter_block().toarray()
A_coup = M.layers.build_coupling_block().toarray()
print('intra + inter + coupling == A?', np.allclose(A_intra + A_inter + A_coup, A.toarray()))


In [ ]:
# Combinatorial Laplacian
L = M.layers.supra_laplacian(kind='comb').toarray()
print('L = D - A:'); print(L)
print('L symmetric?', np.allclose(L, L.T))


In [ ]:
# Random walk transition matrix is row-stochastic
P = M.layers.transition_matrix().toarray()
print('row sums of P:', P.sum(axis=1))


In [ ]:
# Spectral probes
lam2, fiedler = M.layers.algebraic_connectivity()
print('algebraic connectivity (lambda_2):', lam2)
print('Fiedler vector:', fiedler)


In [ ]:
# Layer-aware descriptors
pc = M.layers.participation_coefficient()
vers = M.layers.versatility()
print('participation coefficient:', pc)
print('versatility:', vers)


## 7. Views & subgraphs

`G.views.*` returns lazy, filterable dataframes. The `materialize()` path on a `GraphView`
gives you a concrete subgraph.


In [ ]:
# Edge table view
G.views.edges().head(5)


In [ ]:
# Vertex table view
G.views.vertices().head(5)


In [ ]:
# Slice table view
G.views.slices()


In [ ]:
# Materialize a filtered subgraph
from annnet.core._Views import GraphView

v = GraphView(G, vertices={'A', 'B', 'C'}, edges={'e1', 'e2'})
sub = v.materialize()
print('subgraph nv, ne =', sub.nv, sub.ne)
print('subgraph vertices:', sorted(sub.vertices()))


## 8. IO buffet

Round-trip the graph through several formats. Each branch is independent; cherry-pick
what you need.


In [ ]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp(prefix='annnet_showcase_'))
print('using temp dir:', tmp)


In [ ]:
# Parquet (lossless)
from annnet import to_parquet, from_parquet

to_parquet(G, tmp / 'graph_parquet')
G_back = from_parquet(tmp / 'graph_parquet')
print('parquet round-trip: nv=', G_back.nv, 'ne=', G_back.ne)


In [ ]:
# JSON
from annnet import to_json, from_json

to_json(G, tmp / 'graph.json')
G_json = from_json(tmp / 'graph.json')
print('json round-trip: nv=', G_json.nv, 'ne=', G_json.ne)


In [ ]:
# Native .annnet format (compressed, manifest-aware)
from annnet import read, write

write(G, tmp / 'graph.annnet')
G_native = read(tmp / 'graph.annnet')
print('.annnet round-trip: nv=', G_native.nv, 'ne=', G_native.ne)


In [ ]:
# CX2 (Cytoscape Exchange) — useful for visualization
from annnet import to_cx2, from_cx2

cx2 = to_cx2(G)
G_cx2 = from_cx2(cx2)
print('cx2 round-trip: nv=', G_cx2.nv, 'ne=', G_cx2.ne)


In [ ]:
# DataFrames (per-table)
from annnet import to_dataframes, from_dataframes

dfs = to_dataframes(G)
print('tables exported:', list(dfs.keys()))


## 9. Backend interop

Three backends: NetworkX (`G.nx`), igraph (`G.ig`), graph-tool (`G.gt`, optional).
Each is exposed as an accessor that forwards a familiar API onto a snapshot of the graph.


In [ ]:
# NetworkX accessor: get the underlying nx graph via .backend()
import networkx as nx
nxG = G.nx.backend()
print('nx graph type:', type(nxG).__name__)
print('degree(A):', nxG.degree('A'))
print('shortest_path A->D:', nx.shortest_path(nxG, 'A', 'D'))


In [ ]:
# Peek at vertex IDs the accessor sees
print('peek vertices:', G.nx.peek_vertices(10))


In [ ]:
# igraph accessor (same shape)
igG_view = G.ig.backend()
print('ig graph type:', type(igG_view).__name__)
print('vcount:', igG_view.vcount(), 'ecount:', igG_view.ecount())


In [ ]:
# Export to igraph + manifest, then re-import losslessly
from annnet import to_igraph, from_igraph

ig_graph, manifest = to_igraph(G)
G_via_ig = from_igraph(ig_graph, manifest)
print('igraph round-trip: nv=', G_via_ig.nv, 'ne=', G_via_ig.ne)


## 10. History & replay

`G.history` records every mutating call. Useful for provenance and for replaying the
construction pipeline on a fresh graph.


In [ ]:
# Inspect the call log — G.history() is callable and returns a list of events
log = G.history()
print('total mutating calls:', len(log))
for entry in log[:5]:
    print(' ', entry.get('op'), '-', {k: v for k, v in entry.items() if k in ('args', 'kwargs', 'result')})


In [ ]:
# Tag named snapshots for later diffing
G.history.snapshot('after_construction')
G.attrs.set_vertex_attrs('A', extra='just-added')
G.add_vertices(['E'])
G.history.snapshot('after_extra')

# Each snapshot captures vertex / edge / slice id sets
for snap in G.history.list_snapshots():
    print(snap['label'], '— version:', snap['version'], 'nv:', len(snap['vertex_ids']))


In [ ]:
# Diff two snapshots: what changed between marks?
diff = G.history.diff('after_construction', 'after_extra')
print(diff.summary())


---

That's the tour. Next stops:

- `notebooks/special/sp01_directed_hyperedges.ipynb` — hyperedge stoichiometry deep-dive
- `notebooks/special/sp02_multilayer_math.ipynb` — supra-Laplacian / random walks / modularity
- `notebooks/special/sp03_flexible_edge_orientation.ipynb` — edge direction policies
- `notebooks/use_cases/uc1..uc3` — real-data workflows
- `notebooks/benchmark.ipynb` — tunable speed/memory profiling
